# Module 3: Intent Classifier (LOCAL TRAINING)
Dataset pulled via HuggingFace `datasets` API.
CPU-only, no GPU needed -- runs fast on 9 cores.

Approach: TF-IDF (word n-grams) + Linear SVM, trained on the 27 gold fine-grained
intents, then condensed to 7 routing categories at inference time.

In [1]:
# pip install datasets scikit-learn joblib

import pandas as pd
import numpy as np
import joblib
import json
import os
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score

## Load dataset

In [2]:
ds = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset")
df = ds["train"].to_pandas()
print(df.shape)
print(df["intent"].value_counts())

README.md:   0%|          | 0.00/11.9k [00:00<?, ?B/s]

Bitext_Sample_Customer_Support_Training_(…): reconstructing file:   0%|          |  0.00B / 19.2MB            

Bitext_Sample_Customer_Support_Training_(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/26872 [00:00<?, ? examples/s]

(26872, 5)
intent
check_invoice               1000
complaint                   1000
contact_customer_service    1000
edit_account                1000
switch_account              1000
check_payment_methods        999
contact_human_agent          999
delivery_period              999
get_invoice                  999
newsletter_subscription      999
payment_issue                999
registration_problems        999
cancel_order                 998
place_order                  998
track_refund                 998
change_order                 997
check_refund_policy          997
create_account               997
get_refund                   997
review                       997
set_up_shipping_address      997
delete_account               995
delivery_options             995
recover_password             995
track_order                  995
change_shipping_address      973
check_cancellation_fee       950
Name: count, dtype: int64


## Condense 27 intents -> 7 routing categories

In [3]:
INTENT_TO_ROUTE = {
    "greet": "small_talk", "goodbye": "small_talk", "thank_you": "small_talk",
    "track_order": "order_status", "delivery_options": "order_status", "delivery_period": "order_status",
    "cancel_order": "order_management", "change_order": "order_management", "place_order": "order_management",
    "check_invoice": "billing_and_refunds", "get_refund": "billing_and_refunds", "payment_issue": "billing_and_refunds",
    "get_invoice": "billing_and_refunds", "track_refund": "billing_and_refunds",
    "check_refund_policy": "billing_and_refunds", "check_payment_methods": "billing_and_refunds",
    "create_account": "account_management", "edit_account": "account_management",
    "delete_account": "account_management", "switch_account": "account_management",
    "recover_password": "account_management", "registration_problems": "account_management",
    "complaint": "complaint", "review": "complaint",
    "change_shipping_address": "order_management", "set_up_shipping_address": "order_management",
    "newsletter_subscription": "out_of_scope", "contact_customer_service": "out_of_scope",
    "contact_human_agent": "out_of_scope",
}
ROUTE_CATEGORIES = [
    "small_talk", "order_status", "order_management",
    "billing_and_refunds", "account_management", "complaint", "out_of_scope",
]

def map_intent(intent):
    if intent not in INTENT_TO_ROUTE:
        print(f"WARNING: unmapped intent '{intent}' -> defaulting to out_of_scope")
        return "out_of_scope"
    return INTENT_TO_ROUTE[intent]

df["route_category"] = df["intent"].apply(map_intent)
print(df["route_category"].value_counts())

route_category
billing_and_refunds    6989
account_management     5986
order_management       4963
out_of_scope           3948
order_status           2989
complaint              1997
Name: count, dtype: int64


## Preprocessing + split

In [4]:
def clean_text(s: str) -> str:
    return str(s).strip().lower()

df["text_clean"] = df["instruction"].apply(clean_text)

train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["intent"])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df["intent"])
print(train_df.shape, val_df.shape, test_df.shape)

(21497, 7) (2687, 7) (2688, 7)


## Encode + vectorize

In [5]:
le = LabelEncoder()
y_train = le.fit_transform(train_df["intent"])
y_val = le.transform(val_df["intent"])
y_test = le.transform(test_df["intent"])
fine_label_names = list(le.classes_)

vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=30000, sublinear_tf=True, min_df=2)
X_train = vectorizer.fit_transform(train_df["text_clean"])
X_val = vectorizer.transform(val_df["text_clean"])
X_test = vectorizer.transform(test_df["text_clean"])

## Train

In [6]:
base_svm = LinearSVC(C=1.0, max_iter=5000)
clf = CalibratedClassifierCV(base_svm, cv=3)
clf.fit(X_train, y_train)

,"estimator estimator: estimator instance, default=NoneThe classifier whose output need to be calibrated to provide moreaccurate `predict_proba` outputs. The default classifier isa :class:`~sklearn.svm.LinearSVC`... versionadded:: 1.2",LinearSVC(max_iter=5000)
,"cv cv: int, cross-validation generator, or iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross-validation,- integer, to specify the number of folds,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if ``y`` is binary or multiclass,:class:`~sklearn.model_selection.StratifiedKFold` is used. If ``y`` isneither binary nor multiclass, :class:`~sklearn.model_selection.KFold`is used.Refer to the :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"method method: {'sigmoid', 'isotonic', 'temperature'}, default='sigmoid'The method to use for calibration. Can be:- 'sigmoid', which corresponds to Platt's method (i.e. a binary logistic regression model).- 'isotonic', which is a non-parametric approach.- 'temperature', temperature scaling.Sigmoid and isotonic calibration methods natively support only binaryclassifiers and extend to multi-class classification using a One-vs-Rest (OvR)strategy with post-hoc renormalization, i.e., adjusting the probabilities aftercalibration to ensure they sum up to 1.In contrast, temperature scaling naturally supports multi-class calibration byapplying `softmax(classifier_logits/T)` with a value of `T` (temperature)that optimizes the log loss.For very uncalibrated classifiers on very imbalanced datasets, sigmoidcalibration might be preferred because it fits an additional interceptparameter. This helps shift decision boundaries appropriately when theclassifier being calibrated is biased towards the majority class.Isotonic calibration is not recommended when the number of calibration samplesis too low ``(≪1000)`` since it then tends to overfit... versionchanged:: 1.8 Added option 'temperature'.",'sigmoid'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors.Base estimator clones are fitted in parallel across cross-validationiterations.See :term:`Glossary <n_jobs>` for more details... versionadded:: 0.24",None
,"ensemble ensemble: bool, or ""auto"", default=""auto""Determines how the calibrator is fitted.""auto"" will use `False` if the `estimator` is a:class:`~sklearn.frozen.FrozenEstimator`, and `True` otherwise.If `True`, the `estimator` is fitted using training data, andcalibrated using testing data, for each `cv` fold. The final estimatoris an ensemble of `n_cv` fitted classifier and calibrator pairs, where`n_cv` is the number of cross-validation folds. The output is theaverage predicted probabilities of all pairs.If `False`, `cv` is used to compute unbiased predictions, via:func:`~sklearn.model_selection.cross_val_predict`, which are thenused for calibration. At prediction time, the classifier used is the`estimator` trained on all the data.Note that this method is also internally implemented in:mod:`sklearn.svm` estimators with the `probabilities=True` parameter... versionadded:: 0.24.. versionchanged:: 1.6 `""auto""` option is added and is the default.",'auto'
Name,Type,Value
"calibrated_classifiers_ calibrated_classifiers_: list (len() equal to cv or 1 if `ensemble=False`)The list of classifier and calibrator pairs.- When `ensemble=True`, `n_cv` fitted `estimator` and calibrator pairs. `n_cv` is the number of cross-validation folds.- When `ensemble=False`, the `estimator`, fitted on all the data, and fitted calibrator... versionchanged:: 0.24 Single calibrated classifier case when `ensemble=False`.",list,"[<sklearn.cali...x7f7810149fd0>, <sklearn.

## Evaluate (fine-grained + routing-level)

In [7]:
val_preds = clf.predict(X_val)
print("Validation accuracy (27-class):", accuracy_score(y_val, val_preds))

test_preds = clf.predict(X_test)
print("Test accuracy (27-class):", accuracy_score(y_test, test_preds))

test_pred_intents = le.inverse_transform(test_preds)
test_pred_routes = [map_intent(i) for i in test_pred_intents]
test_true_routes = test_df["route_category"].tolist()
print("Test accuracy (7-class routing):", accuracy_score(test_true_routes, test_pred_routes))
print(classification_report(test_true_routes, test_pred_routes, labels=ROUTE_CATEGORIES))

Validation accuracy (27-class): 0.9944175660588016
Test accuracy (27-class): 0.9955357142857143
Test accuracy (7-class routing): 0.9981398809523809
                     precision    recall  f1-score   support

         small_talk       0.00      0.00      0.00         0
       order_status       1.00      0.99      1.00       300
   order_management       0.99      1.00      1.00       495
billing_and_refunds       1.00      1.00      1.00       700
 account_management       1.00      1.00      1.00       598
          complaint       1.00      1.00      1.00       200
       out_of_scope       1.00      1.00      1.00       395

           accuracy                           1.00      2688
          macro avg       0.86      0.86      0.86      2688
       weighted avg       1.00      1.00      1.00      2688



/home/alihamdi/Downloads/ecommerce-chatbot-local/local_app/venv/lib64/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/alihamdi/Downloads/ecommerce-chatbot-local/local_app/venv/lib64/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/alihamdi/Downloads/ecommerce-chatbot-local/local_app/venv/lib64/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `z

## Save artifacts directly into local_app/models/intent/

In [8]:
OUTPUT_DIR = "../local_app/models/intent"
os.makedirs(OUTPUT_DIR, exist_ok=True)

joblib.dump(vectorizer, os.path.join(OUTPUT_DIR, "intent_vectorizer.pkl"))
joblib.dump(clf, os.path.join(OUTPUT_DIR, "intent_model.pkl"))
with open(os.path.join(OUTPUT_DIR, "intent_fine_labels.json"), "w") as f:
    json.dump({str(i): lbl for i, lbl in enumerate(fine_label_names)}, f, indent=2)
with open(os.path.join(OUTPUT_DIR, "intent_to_route.json"), "w") as f:
    json.dump(INTENT_TO_ROUTE, f, indent=2)

print("Saved to", OUTPUT_DIR)
print(os.listdir(OUTPUT_DIR))

Saved to ../local_app/models/intent
['.gitkeep', 'intent_vectorizer.pkl', 'intent_model.pkl', 'intent_fine_labels.json', 'intent_to_route.json']
